In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

# key.env is not the default filename, so load it explicitly
load_dotenv("key.env")

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

BASE_DIR = Path.cwd()
AUDIO_DIR = BASE_DIR  # you're keeping MS072.mp3 in the root — that's fine
TRANSCRIPTS_DIR = BASE_DIR / "transcripts"
CHUNKS_DIR = BASE_DIR / "chunks"

for directory in (TRANSCRIPTS_DIR, CHUNKS_DIR):
    directory.mkdir(exist_ok=True)

print("Client initialized:", client.api_key is not None)
print("Transcripts dir:", TRANSCRIPTS_DIR)
print("Chunks dir:", CHUNKS_DIR)

Client initialized: True
Transcripts dir: c:\Users\anura\Desktop\Ironhack\Week-2\Whisper-STT\transcripts
Chunks dir: c:\Users\anura\Desktop\Ironhack\Week-2\Whisper-STT\chunks


In [2]:
# ── Step 2: Load the sample audio ─────────────────────────────────────
AUDIO_FILE = BASE_DIR / "MS072.mp3"

assert AUDIO_FILE.exists(), f"Audio file not found: {AUDIO_FILE}"
print("Audio file found:", AUDIO_FILE)
print("File size (MB):", round(AUDIO_FILE.stat().st_size / (1024 * 1024), 2))

Audio file found: c:\Users\anura\Desktop\Ironhack\Week-2\Whisper-STT\MS072.mp3
File size (MB): 2.03


In [3]:
# ── Step 3: Basic transcription (no chunking) ──────────────────────────
with open(AUDIO_FILE, "rb") as audio_file:
    transcript = client.audio.transcriptions.create(
        model="whisper-1",
        file=audio_file
    )

print(transcript.text)

Now you just pretend you hadn't talked to me any before, and you tell me about the government moving you up here. Well, now I don't remember the day or the month or nothing like that, you understand? But I know they come one evening, and I just was fixing to quit working in my garden. And he comes, and he says, well, says, is this Miss Davis? And I said, that's what they call me. Other words ain't new, I suggest. He says, well, since I'm taking pictures around, says the government's going to take over everything, and Gainesville's going to have to move out. I said, all this many years? I said, you talk crazy. And he lay, he says, no. Says, I'm here just to tell you, says, friendly talk. And he says, then my chicken was all out, and it was late in the evening, you know. I told him, I says, well, you'll have to excuse me. I got to put my chickens in the chicken yard. He says, that's just what I want you to do. Well, there was a shark fellow with him, you know. But he didn't ever tell me 

In [4]:
# ── Step 4: Prompted (guided) transcription ─────────────────────────────
context_prompt = (
    "This is an oral history interview about a family's relocation "
    "when the government took over their land. Speaker names may include "
    "Miss Davis. The interview mentions chickens, a chicken yard, and "
    "a photographer taking pictures."
)

with open(AUDIO_FILE, "rb") as audio_file:
    transcript_prompted = client.audio.transcriptions.create(
        model="whisper-1",
        file=audio_file,
        prompt=context_prompt
    )

print(transcript_prompted.text)


Now you just pretend you hadn't talked to me any before and you tell me about the government moving you up here. Other words ain't new, I suggest. He says, well, since I'm taking pictures around, says the government's going to take over everything and Gainesville's going to have to move out. I said, all this many years? I said, you talk crazy. And he lay, he says, no, says I'm here just to tell you, says friendly talk. And he says, then my chickens was all out and it was late in the evening, you know. I told him, I says, well, you'll have to excuse me, I got to put my chickens in the chicken yard. He says, that's just what I want you to do. Well, there was a shark feller with him, you know, but he didn't ever tell me who he was, you understand? And I went to put my chickens in the chicken yard and fed them. And I had on my old bonnet and shirt where I'd been working, you know. And I throwed the corn out and I noticed this shark feller walked up close to him and said something or anothe

In [5]:
# ── Step 5: Compare prompted vs. unprompted ─────────────────────────────
print("Unprompted length (words):", len(transcript.text.split()))
print("Prompted length (words):  ", len(transcript_prompted.text.split()))

Unprompted length (words): 323
Prompted length (words):   269


In [6]:
# ── Optional: retry with a shorter, keyword-style prompt ───────────────
context_prompt_v2 = "Miss Davis, Gainesville, chicken yard, chickens"

with open(AUDIO_FILE, "rb") as audio_file:
    transcript_prompted_v2 = client.audio.transcriptions.create(
        model="whisper-1",
        file=audio_file,
        prompt=context_prompt_v2
    )

print(transcript_prompted_v2.text)
print("Word count:", len(transcript_prompted_v2.text.split()))

Now you just pretend you hadn't talked to me any before, and you tell me about the government moving you up here. Well, now I don't remember the day or the month or nothing like that, you understand? But I know they come one evening, and I just was fixing to quit working in my garden. And he comes and he says, well, says, is this Miss Davis? And I said, that's what they call me. Other words ain't new, I suggest. He says, well, since I'm taking pictures around, says the government's going to take over everything and Gainesville's going to have to move out. I said, all this many years? I said, you talk crazy. And he lay, he says, no, says, I'm here just to tell you, says, friendly talk. And he says, then my chickens was all out and it was late in the evening, you know. I told him, I says, well, you'll have to excuse me, I got to put my chickens in the chicken yard. He says, that's just what I want you to do. Well, there was a shark fellow with him, you know. But he didn't ever tell me wh

In [7]:
# ── Step 6: Split audio into chunks ─────────────────────────────────────
from pydub import AudioSegment

CHUNK_LENGTH_MINUTES = 10

audio = AudioSegment.from_file(AUDIO_FILE)
duration_ms = len(audio)
chunk_length_ms = CHUNK_LENGTH_MINUTES * 60 * 1000

chunk_paths = []
for i, start_ms in enumerate(range(0, duration_ms, chunk_length_ms)):
    end_ms = min(start_ms + chunk_length_ms, duration_ms)
    chunk = audio[start_ms:end_ms]
    chunk_path = CHUNKS_DIR / f"chunk_{i:03d}.mp3"
    chunk.export(chunk_path, format="mp3")
    chunk_paths.append({"path": chunk_path, "start_ms": start_ms, "end_ms": end_ms})
    print(f"Created {chunk_path.name}: {start_ms/1000:.1f}s → {end_ms/1000:.1f}s")

print(f"\nTotal duration: {duration_ms/1000:.1f}s across {len(chunk_paths)} chunk(s)")

Created chunk_000.mp3: 0.0s → 133.0s

Total duration: 133.0s across 1 chunk(s)


In [ ]:
import sys
print(sys.executable)  # sanity check — should show the envs\lab path
!{sys.executable} -m pip install pydub

In [10]:
import sys
!{sys.executable} -m pip install audioop-lts

In [8]:
# ── Demonstrate multi-chunk splitting (temporary, for submission evidence) ──
CHUNK_LENGTH_MINUTES_DEMO = 0.5  # 30 seconds — forces multiple chunks on a 133s file

chunk_length_ms_demo = int(CHUNK_LENGTH_MINUTES_DEMO * 60 * 1000)
demo_chunks = []
for i, start_ms in enumerate(range(0, duration_ms, chunk_length_ms_demo)):
    end_ms = min(start_ms + chunk_length_ms_demo, duration_ms)
    chunk = audio[start_ms:end_ms]
    print(f"[DEMO] chunk_{i:03d}: {start_ms/1000:.1f}s → {end_ms/1000:.1f}s")
    demo_chunks.append({"start_ms": start_ms, "end_ms": end_ms})

print(f"\n[DEMO] {len(demo_chunks)} chunks from a 30s chunk length")

[DEMO] chunk_000: 0.0s → 30.0s
[DEMO] chunk_001: 30.0s → 60.0s
[DEMO] chunk_002: 60.0s → 90.0s
[DEMO] chunk_003: 90.0s → 120.0s
[DEMO] chunk_004: 120.0s → 133.0s

[DEMO] 5 chunks from a 30s chunk length


In [9]:
# ── Step 7: Transcribe chunks with timestamps ────────────────────────────
all_segments = []

for chunk_info in chunk_paths:
    with open(chunk_info["path"], "rb") as f:
        result = client.audio.transcriptions.create(
            model="whisper-1",
            file=f,
            response_format="verbose_json",
            timestamp_granularities=["segment"]
        )

    offset_seconds = chunk_info["start_ms"] / 1000

    for seg in result.segments:
        all_segments.append({
            "start": seg.start + offset_seconds,
            "end": seg.end + offset_seconds,
            "text": seg.text.strip()
        })

print(f"Total segments across all chunks: {len(all_segments)}")
for seg in all_segments[:5]:
    print(f"[{seg['start']:.1f}s - {seg['end']:.1f}s] {seg['text']}")

Total segments across all chunks: 27
[0.0s - 6.0s] Now you just pretend you hadn't talked to me any before, and you tell me about the government moving you up here.
[6.0s - 12.0s] Well, now, I don't remember the day or the month or nothing like that, you understand?
[12.0s - 19.0s] But I know they come one evening, and I just was fixing to quit working in my garden.
[19.0s - 27.0s] And he comes and he says, well, says, is this Miss Davis?
[27.0s - 30.0s] And I said, that's what they called me.


In [10]:
# ── Step 8: Export with timestamps ───────────────────────────────────────
import json

def format_srt_timestamp(seconds):
    """Convert seconds to SRT timestamp format: HH:MM:SS,mmm"""
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    millis = int((seconds % 1) * 1000)
    return f"{hours:02d}:{minutes:02d}:{secs:02d},{millis:03d}"

# 1. Human-readable TXT
txt_path = TRANSCRIPTS_DIR / "MS072_transcript.txt"
with open(txt_path, "w", encoding="utf-8") as f:
    for seg in all_segments:
        f.write(f"[{seg['start']:.1f}s - {seg['end']:.1f}s] {seg['text']}\n")
print(f"Saved: {txt_path}")

# 2. SRT (subtitle format)
srt_path = TRANSCRIPTS_DIR / "MS072_transcript.srt"
with open(srt_path, "w", encoding="utf-8") as f:
    for i, seg in enumerate(all_segments, start=1):
        f.write(f"{i}\n")
        f.write(f"{format_srt_timestamp(seg['start'])} --> {format_srt_timestamp(seg['end'])}\n")
        f.write(f"{seg['text']}\n\n")
print(f"Saved: {srt_path}")

# 3. JSON
json_path = TRANSCRIPTS_DIR / "MS072_transcript.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(all_segments, f, indent=2, ensure_ascii=False)
print(f"Saved: {json_path}")

Saved: c:\Users\anura\Desktop\Ironhack\Week-2\Whisper-STT\transcripts\MS072_transcript.txt
Saved: c:\Users\anura\Desktop\Ironhack\Week-2\Whisper-STT\transcripts\MS072_transcript.srt
Saved: c:\Users\anura\Desktop\Ironhack\Week-2\Whisper-STT\transcripts\MS072_transcript.json
